In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU, then rerun"
print("GPU:", torch.cuda.get_device_name(0))

!pip -q install "tensorrt>=10" ultralytics onnx onnxslim fastapi uvicorn python-multipart requests

from ultralytics import YOLO
YOLO("yolov8n.pt").export(format="engine", half=True, dynamic=True, batch=8, imgsz=640)
!ls -lh yolov8n.*

GPU: Tesla T4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ TensorRT requires GPU export, automatically assigning device=0
Ultralytics 8.4.210 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
💡 ProTip: Export to OpenVINO format for 

2026-11-11 05:13:52,822 - [modelopt][onnx] - INFO - Successfully enabled 1 EPs for ORT: ['CPUExecutionProvider']


INFO:modelopt.onnx:Successfully enabled 1 EPs for ORT: ['CPUExecutionProvider']


2026-11-11 05:13:53,273 - [modelopt][onnx] - INFO - Running ONNX Runtime to obtain reference outputs (this may take a while)...


INFO:modelopt.onnx.autocast:Running ONNX Runtime to obtain reference outputs (this may take a while)...


2026-11-11 05:14:01,707 - [modelopt][onnx] - WARNING - Did not find  in value info map! Assuming not castable


2026-11-11 05:14:03,068 - [modelopt][onnx] - INFO - Converted 315/321 nodes (98.13%) to fp16


INFO:modelopt.onnx.autocast:Converted 315/321 nodes (98.13%) to fp16


TensorRT: input "images" with shape(-1, 3, -1, -1) DataType.FLOAT
TensorRT: output "output0" with shape(-1, 84, -1) DataType.FLOAT
TensorRT: building FP16 engine as yolov8n.engine
TensorRT: export success ✅ 174.2s, saved as 'yolov8n.engine' (8.0 MB)

Export complete (176.2s)
Results saved to /content/yolov8n.engine
Predict:         yolo predict task=detect model=yolov8n.engine imgsz=640 quantize=16
Validate:        yolo val task=detect model=yolov8n.engine imgsz=640 data=coco.yaml quantize=16
Visualize:       https://netron.app
-rw-r--r-- 1 root root 8.1M Nov 11 05:16 yolov8n.engine
-rw-r--r-- 1 root root 6.4M Nov 11 05:14 yolov8n.fp16.onnx
-rw-r--r-- 1 root root  13M Nov 11 05:13 yolov8n.onnx
-rw-r--r-- 1 root root 6.3M Nov 11 05:13 yolov8n.pt


In [ ]:
%%writefile app.py
import io, time, threading
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import HTMLResponse, Response
from PIL import Image
from ultralytics import YOLO

app = FastAPI(title="VisionPort")
model = YOLO("yolov8n.engine", task="detect")
lock = threading.Lock()

PAGE = """
<!doctype html>
<html><head><meta charset="utf-8"><title>VisionPort</title>
<style>
 body{font-family:system-ui,sans-serif;max-width:900px;margin:30px auto;padding:0 16px}
 img{max-width:100%;margin-top:16px;border-radius:8px}
 button{padding:8px 18px;font-size:16px;cursor:pointer}
</style></head>
<body>
<h2>VisionPort: object detection</h2>
<input type="file" id="f" accept="image/*">
<button onclick="go()">Detect</button>
<p id="s"></p>
<img id="out">
<script>
async function go(){
  const file = document.getElementById('f').files[0];
  if(!file){ alert('Choose an image first'); return; }
  document.getElementById('s').textContent = 'Running...';
  const fd = new FormData(); fd.append('file', file);
  const r = await fetch('/detect/image', {method:'POST', body: fd});
  if(!r.ok){ document.getElementById('s').textContent = 'Error ' + r.status; return; }
  document.getElementById('s').textContent = 'GPU inference: ' + r.headers.get('X-Latency-ms') + ' ms | objects: ' + r.headers.get('X-Count');
  document.getElementById('out').src = URL.createObjectURL(await r.blob());
}
</script>
</body></html>
"""

@app.get("/", response_class=HTMLResponse)
def home():
    return PAGE

@app.get("/health")
def health():
    return {"status": "ok"}

def run(img, conf):
    t = time.perf_counter()
    with lock:
        r = model.predict(img, conf=conf, imgsz=640, half=True, verbose=False)[0]
    return r, (time.perf_counter() - t) * 1000

@app.post("/detect")
def detect(file: UploadFile = File(...), conf: float = 0.25):
    img = Image.open(io.BytesIO(file.file.read())).convert("RGB")
    r, ms = run(img, conf)
    dets = [
        {"label": r.names[int(b.cls)], "conf": round(float(b.conf), 3),
         "box": [round(v, 1) for v in b.xyxy[0].tolist()]}
        for b in r.boxes
    ]
    return {"detections": dets, "latency_ms": round(ms, 1)}

@app.post("/detect/image")
def detect_image(file: UploadFile = File(...), conf: float = 0.25):
    img = Image.open(io.BytesIO(file.file.read())).convert("RGB")
    r, ms = run(img, conf)
    out = Image.fromarray(r.plot()[..., ::-1])  # BGR -> RGB
    buf = io.BytesIO()
    out.save(buf, format="JPEG", quality=90)
    return Response(buf.getvalue(), media_type="image/jpeg",
                    headers={"X-Latency-ms": f"{ms:.1f}", "X-Count": str(len(r.boxes))})

Overwriting app.py


In [ ]:
import subprocess, time, requests, re, os

!pkill -f uvicorn
!pkill -f cloudflared
time.sleep(3)

server = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=open("server.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(60):
    try:
        if requests.get("http://127.0.0.1:8000/health").ok:
            print("Server is up")
            break
    except Exception:
        time.sleep(2)
else:
    print(open("server.log").read())

if not os.path.exists("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                          stdout=subprocess.DEVNULL, stderr=open("tunnel.log", "w"))
for _ in range(30):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        print("\nOpen this link:", m.group(0))
        break
else:
    print(open("tunnel.log").read())

Server is up

Open this link: https://retrieved-investigator-intelligence-tell.trycloudflare.com
